# Transit times from a Gravitas light curve

A notebook the bridge runs (spike). It reads one `gravitas.observation/1` time series, handed to it as `gravitas_input`, and nothing else: the columns by their roles, their units, the masks. It finds each transit, measures its mid-time and depth, fits a period, and returns a table and a folded plot as `gravitas_output`.


In [ ]:
import json
import numpy as np

obs = json.loads(gravitas_input)
assert obs["format"] == "gravitas.observation" and obs["formatVersion"] == 1
assert obs["kind"] == "time-series"

def column(role, of=None):
    for c in obs["columns"]:
        if c["role"] == role and (of is None or c.get("of") == of):
            return c
    return None

xc = next(c for c in obs["columns"] if c["id"] == obs["time"]["column"])
yc = column("value")
ec = column("uncertainty", of=yc["id"])
num = lambda c: np.array([np.nan if v is None else v for v in c["values"]], float)
t, f = num(xc), num(yc)
e = num(ec) if ec else np.full_like(f, np.nan)
masked = np.zeros(len(t), bool)
for m in obs["masks"]:
    masked[m["rows"]] = True
keep = np.isfinite(t) & np.isfinite(f) & ~masked
t, f, e = t[keep], f[keep], e[keep]
level = np.median(f)
f, e = f / level, e / level
print(f"{len(t)} points from {t.min():.3f} to {t.max():.3f} {xc['unit']} ({obs['time']['format']}, {obs['time']['scale']})")


In [ ]:
# A transit is a run of points well below the out-of-transit level.
scatter = 1.4826 * np.median(np.abs(f - np.median(f)))
low = f < 1 - 5 * scatter
runs, start = [], None
for i, is_low in enumerate(low):
    if is_low and start is None:
        start = i
    if start is not None and (not is_low or i == len(low) - 1 or t[i + 1] - t[i] > 0.1):
        end = i if is_low else i - 1
        if end - start >= 4:
            runs.append((start, end))
        start = None

transits = []
for a, b in runs:
    tt, ff = t[a:b + 1], f[a:b + 1]
    q = (tt > tt[0] + 0.25 * (tt[-1] - tt[0])) & (tt < tt[-1] - 0.25 * (tt[-1] - tt[0]))
    mid = 0.5 * (tt[0] + tt[-1])
    depth = 1 - ff[q].mean()
    depth_err = ff[q].std(ddof=1) / np.sqrt(q.sum())
    transits.append((mid, depth, depth_err, int(q.sum())))
print(len(transits), "transits")


In [ ]:
# The period: mid-times against transit number.
mids = np.array([m for m, *_ in transits])
gap = np.min(np.diff(mids))
n = np.round((mids - mids[0]) / gap).astype(int)
(period, t0), cov = np.polyfit(n, mids, 1, cov=True)
period_err = float(np.sqrt(cov[0, 0]))
depths = np.array([d for _, d, _, _ in transits])
print(f"period {period:.5f} ± {period_err:.5f} d; mean depth {1e6 * depths.mean():.0f} ppm")


In [ ]:
import io, base64
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

phase = ((t - t0 + 0.5 * period) % period - 0.5 * period) * 24
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=100)
ax.plot(phase, 1e6 * (f - 1), ".", ms=2, color="0.3")
ax.set_xlim(-4, 4)
ax.set_xlabel("hours from mid-transit")
ax.set_ylabel("flux - 1 (ppm)")
ax.set_title(f"{obs['title']}: folded at {period:.4f} d")
fig.tight_layout()
png = io.BytesIO()
# No date or version in the file, so the same run gives the same bytes.
fig.savefig(png, format="png", metadata={"Software": None})
plt.close(fig)

rows = len(transits)
gravitas_output = {
    "table": {
        "format": "gravitas.observation", "formatVersion": 1, "kind": "table",
        "id": f"{obs['id']}:transits", "title": f"{obs['title']}: transit times and depths",
        "object": obs["object"], "facility": obs["facility"], "origin": "model",
        "source": {"kind": "import", "id": obs["id"], "version": obs["source"].get("version")},
        "credit": obs["credit"], "license": obs["license"], "retrieved": obs["retrieved"],
        "citations": obs["citations"],
        "reductions": obs["reductions"] + [
            "Normalized by the median; a transit is five or more consecutive points 5 robust standard deviations below it.",
            "Mid-time is the center of the run; depth is the mean of its central half. Period and epoch from a straight line through the mid-times.",
        ],
        "axes": {"x": "transit", "y": "depth"},
        "columns": [
            {"id": "transit", "name": "transit", "unit": "", "role": "x", "values": [int(k) for k in n]},
            {"id": "mid", "name": "mid-time", "unit": xc["unit"], "role": "value", "values": [float(m) for m in mids]},
            {"id": "depth", "name": "depth", "unit": "", "role": "value", "values": [float(d) for d in depths]},
            {"id": "depth-error", "name": "depth uncertainty", "unit": "", "role": "uncertainty", "of": "depth",
             "values": [float(x) for _, _, x, _ in transits]},
        ],
        "masks": [], "annotations": [],
    },
    "values": {"period": float(period), "periodError": period_err, "epoch": float(t0), "unit": xc["unit"]},
    "figure": {"type": "image/png", "base64": base64.b64encode(png.getvalue()).decode("ascii")},
}
print(rows, "rows returned")
